## Imports

In [1]:
from pynq.lib.video import *
import numpy as np
import threading
import time
import cv2
from pynq.overlays.base import BaseOverlay
from datetime import datetime
base = BaseOverlay("base.bit")
btns = base.btns_gpio

## Functions and set up

In [2]:
%%microblaze base.PMODB

#include "gpio.h"
#include "pyprintf.h"

//Function to turn on/off a selected pin of PMODB
int write_gpio(unsigned int pin, unsigned int val){
    if (val > 1){
        pyprintf("pin value must be 0 or 1");
    }
    gpio pin_out = gpio_open(pin);
    gpio_set_direction(pin_out, GPIO_OUT);
    gpio_write(pin_out, val);
    return 0;
}

//Function to read the value of a selected pin of PMODB
unsigned int read_gpio(unsigned int pin){
    gpio pin_in = gpio_open(pin);
    gpio_set_direction(pin_in, GPIO_IN);
    return gpio_read(pin_in);
}

void clear_gpios(){
    for(int i = 0; i < 8; ++i){
        write_gpio(i,0);
    }
}

In [3]:
clear_gpios()

In [4]:
def write_gpio_pwm(pin, brightness, d=0.5, freq=200):
    """
    pin: RBG -> pins 2, 1, 3 respectively
    brightness: 0–100
    duration: time LED stays on (s)
    freq: PWM frequency (Hz)
    """
    if brightness < 0: brightness = 0
    if brightness > 100: brightness = 100

    period = 1.0 / freq
    on_time = period * (brightness / 100.0)
    off_time = period - on_time

    end_time = time.time() + d

    while time.time() < end_time:
        write_gpio(pin, 1)
        time.sleep(on_time)
        write_gpio(pin, 0)
        time.sleep(off_time)
    write_gpio(pin, 0)

In [5]:
# set led functions, incorporate this to sample the state every 0.1s
def set_led_blue(duration=0.1):
    write_gpio_pwm(1,50,d=duration)

def set_led_red(duration=0.1):
    write_gpio_pwm(2,50,d=duration)

def set_led_green(duration=0.1):
    write_gpio_pwm(3,50,d=duration)

In [6]:
def get_region(frame, x1, y1, x2, y2):
    return {
        "box": (x1, y1, x2, y2),
        "region": frame[y1:y2, x1:x2]
    }

In [7]:
# get color of skin, hair, eyes, etc
def get_color(region):
    pixels = region.reshape((-1, 3))
    
    if len(pixels) == 0:
        return (0, 0, 0)

    avg_color = np.mean(pixels, axis=0)
    return tuple(avg_color.astype(int))

In [8]:
def color_correct(frame):

    b, g, r = cv2.split(frame)
    b_avg, g_avg, r_avg = np.mean(b), np.mean(g), np.mean(r)

    k = (r_avg + g_avg + b_avg) / 3.0

    r = np.clip(r * k/r_avg, 0, 255).astype(np.uint8)
    g = np.clip(g * k/g_avg, 0, 255).astype(np.uint8)
    b = np.clip(b * k/b_avg, 0, 255).astype(np.uint8)

    frame_wb = cv2.merge((b, g, r))

    lab = cv2.cvtColor(frame_wb, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    l_norm = cv2.normalize(l, None, 0, 255, cv2.NORM_MINMAX)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l_clahe = clahe.apply(l_norm)

    lab_corrected = cv2.merge((l_clahe, a, b))

    return cv2.cvtColor(lab_corrected, cv2.COLOR_LAB2BGR)

In [9]:
def extract_regions(frame, x, y, w, h):

    cheek = get_region(
        frame,
        x + int(0.15*w),
        y + int(0.55*h),
        x + int(0.40*w),
        y + int(0.85*h)
    )

    hair = get_region(
        frame,
        x + int(0.30*w),
        max(0, y - int(0.20*h)),
        x + int(0.70*w),
        y - int(0.03*h)
    )

    eye = get_region(
        frame,
        x + w//10,
        y + h//5,
        x + w//2,
        y + 2*h//5
    )

    return cheek, hair, eye

In [10]:
def extract_colors(cheek_region, hair_region, eye_region):

    # skin:
    hsv_cheek = cv2.cvtColor(cheek_region, cv2.COLOR_BGR2HSV)

    lower_skin = (0,30,60)
    upper_skin = (20,150,255)

    skin_mask = cv2.inRange(hsv_cheek, lower_skin, upper_skin)

    skin_only = cv2.bitwise_and(
        cheek_region,
        cheek_region,
        mask=skin_mask
    )

    
    # hair:
    hsv_hair = cv2.cvtColor(hair_region, cv2.COLOR_BGR2HSV)

    skin_mask = cv2.inRange(hsv_hair, lower_skin, upper_skin)

    hair_only = cv2.bitwise_and(
        hair_region,
        hair_region,
        mask=cv2.bitwise_not(skin_mask)
    )

    
    # eye:
    hsv_eye = cv2.cvtColor(eye_region, cv2.COLOR_BGR2HSV)

    lower_eye = (0,0,0)
    upper_eye = (180,255,60)

    mask_eye = cv2.inRange(hsv_eye, lower_eye, upper_eye)

    eye_only = cv2.bitwise_and(
        eye_region,
        eye_region,
        mask=mask_eye
    )

    
    return skin_only, hair_only, eye_only

## Threads

In [11]:
face_cascade = cv2.CascadeClassifier(
    "/home/xilinx/jupyter_notebooks/base/video/data/haarcascade_frontalface_default.xml"
)

def face_thread():
    print("[FACE] Thread starting")
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("[FACE] Camera failed")
        return

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    time.sleep(1)

    while True:
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.5)
            continue

        gray_small = cv2.cvtColor(cv2.resize(frame, (320, 240)), cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(
            gray_small,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(60, 60)
        )

        with lock:
            shared["face_count"] = len(faces)

            # check if led set the capture bit
            if shared.get("capture_request", False):
                shared["captured_frame"] = frame.copy()
                shared["capture_request"] = False
                print("[FACE] Frame captured on request")
                break

        time.sleep(0.05)

    cap.release()
    print("[FACE] Thread exiting")

In [12]:
def led_thread():
    print("[LED] Thread starting")
    single_face_start = None

    while True:
        with lock:
            count = shared.get("face_count", 0)

        # no face
        if count == 0:
            set_led_red(0.1)
            single_face_start = None

        # more than 1 face
        elif count > 1:
            set_led_blue(0.1)
            single_face_start = None

        # exactly 1 face
        elif count == 1:
            set_led_green(0.1)
            if single_face_start is None:
                single_face_start = time.time()
            elif time.time() - single_face_start >= 2:
                print("[LED] Stable face for 2 seconds")

                # indicate about to take picture
                for _ in range(3):
                    clear_gpios()
                    time.sleep(0.2)
                    set_led_green(0.2)
                    time.sleep(0.2)

                # set signal bit to camera
                with lock:
                    shared["capture_request"] = True

                print("[LED] Capture requested, exiting LED thread")
                break

        time.sleep(0.1)

    print("[LED] Thread exiting")

## MAIN

In [14]:
if __name__ == "__main__":
    # thread set up
    shared = {
        "face_count": 0,
        "capture_request": False,
        "capture_done": False,
        "captured_frame": None
    }

    lock = threading.Lock()

    t_face = threading.Thread(target=face_thread)
    t_led = threading.Thread(target=led_thread)

    t_face.start()
    t_led.start()

    t_led.join()
    t_face.join()

    print("[MAIN] Threads joined")

# process captured frame
frame = shared.get("captured_frame", None)
cv2.imwrite("original_picture.jpg", frame)

if frame is not None:

    frame_corrected = color_correct(frame)
    cv2.imwrite("captured_corrected.jpg", frame_corrected)
    print("[MAIN] Captured image color-corrected and saved as 'captured_corrected.jpg'")

    # ------------------------------------------------------------------
    # processing part

    print("[MAIN] Processing captured frame...")
    
    frame = frame_corrected
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.3, 5)

    if len(faces) > 0:
        x, y, w, h = faces[0]
        
        cheek, hair, eye = extract_regions(frame_corrected, x,y,w,h)
        
        cheek_region = cheek["region"]
        hair_region = hair["region"]
        eye_region = eye["region"]

        skin_only, hair_only, eye_only = extract_colors(
            cheek_region,
            hair_region,
            eye_region
        )
            
        print("Skin Color:", get_color(skin_only))
        print("Hair Color:", get_color(hair_only))
        print("Eye Color:", get_color(eye_only))

        # ------------------------------------------------------------------
        # debug images
        cv2.imwrite("debug_skin.jpg", cheek_region)
        cv2.imwrite("debug_hair.jpg", hair_region)
        cv2.imwrite("debug_eye.jpg", eye_region)
        
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)

        for region, color in [
            (cheek, (255,0,0)),
            (hair, (0,0,255)),
            (eye, (255,255,0))
        ]:
            x1, y1, x2, y2 = region["box"]
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)

        cv2.imwrite("captured_with_boxes.jpg", frame)
        print("[MAIN] Image saved as captured_with_boxes.jpg")

    else:
        print("[MAIN] No face found in captured frame")

else:
    print("[MAIN] No frame captured")

print("Done.")

[FACE] Thread starting
[LED] Thread starting


[ WARN:2] global ./modules/videoio/src/cap_gstreamer.cpp (616) isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created


[LED] Stable face for 2 seconds
[LED] Capture requested, exiting LED thread
[LED] Thread exiting
[FACE] Frame captured on request
[FACE] Thread exiting
[MAIN] Threads joined
[MAIN] Captured image color-corrected and saved as 'captured_corrected.jpg'
[MAIN] Processing captured frame...
Skin Color: (109, 120, 156)
Hair Color: (45, 45, 44)
Eye Color: (5, 5, 6)


NameError: name 'chcheek_regioneek' is not defined